# QKD BB84 Phase Control

Control notebook for the QKD phase-encoded BB84 system on the RFSoC 4x2.

## Register Map

| Address | Name | R/W | Description |
|---------|------|-----|-------------|
| 0x00 | CTRL | R/W | `[0]` global_en, `[1]` alice_en, `[2]` bob_en |
| 0x04 | ALICE_PHASE_STAGED | R/W | `[1:0]` Staged Alice phase |
| 0x08 | BOB_PHASE_STAGED | R/W | `[1:0]` Staged Bob phase |
| 0x0C | STATUS | R | `[0]` alice_running, `[1]` bob_running, `[2]` sw_mode |
| 0x10 | PHASE_APPLY | R/W | Write 1 to latch both staged phases (auto-clears) |
| 0x14 | ALICE_PHASE_ACTIVE | R | `[1:0]` Active Alice phase |
| 0x18 | BOB_PHASE_ACTIVE | R | `[1:0]` Active Bob phase |
| 0x1C | VERSION | R | 0x2026_0425 |

## Phase Encoding

| Value | Phase | Basis | Bit |
|-------|-------|-------|-----|
| 0b00 | 0 | Z | 0 |
| 0b01 | pi/2 | X | 0 |
| 0b10 | pi | Z | 1 |
| 0b11 | 3pi/2 | X | 1 |

## 1. Load Overlay and Initialize Hardware

In [17]:
from pynq import PL
PL.reset()

import xrfdc
import xrfclk
from pynq import Overlay, MMIO, Clocks
import pprint

# Load the bitstream — update path to match your build output
xrfclk.set_ref_clks()
ol = Overlay('./qkd_phase_bb84.bit')

# Show all IP in the design
pprint.pprint(ol.ip_dict)

RuntimeError: Frequency 122.88 MHz is not valid.

In [2]:
# Get handles to the QKD wrapper and RF data converter
# NOTE: update these names to match your block design instance names
# qkd = ol.qkd_top_wrapper_bd_0
qkd = ol.ip_dict['qkd_top_wrapper_bd_0']
base_addr = ol.ip_dict['qkd_top_wrapper_bd_0']['phys_addr']
addr_range = ol.ip_dict['qkd_top_wrapper_bd_0']['addr_range']
print(f"Base: {base_addr:#010x}, Range: {addr_range:#x}")
qkd_mmio = MMIO(base_addr, addr_range)
print(f"VERSION: {qkd_mmio.read(0x1C):#010x}")

rf = ol.usp_rf_data_converter_0
print(f"RF Data Converter: {rf}")

Base: 0x80000000, Range: 0x1000
VERSION: 0x20260425
RF Data Converter: <xrfdc.RFdc object at 0xffff7c152350>


## 2. Register Access Helpers

In [3]:
# Register offsets
REG_CTRL                = 0x00
REG_ALICE_PHASE_STAGED  = 0x04
REG_BOB_PHASE_STAGED    = 0x08
REG_STATUS              = 0x0C
REG_PHASE_APPLY         = 0x10
REG_ALICE_PHASE_ACTIVE  = 0x14
REG_BOB_PHASE_ACTIVE    = 0x18
REG_VERSION             = 0x1C

PHASE_LABELS = {0: '0', 1: 'pi/2', 2: 'pi', 3: '3pi/2'}

def reg_read(offset):
    return qkd_mmio.read(offset)

def reg_write(offset, value):
    qkd_mmio.write(offset, value)

def set_phases(alice_phase, bob_phase):
    """Stage and atomically apply phase values for both channels.
    
    Args:
        alice_phase: 0-3 (0=0, 1=pi/2, 2=pi, 3=3pi/2)
        bob_phase:   0-3 (same encoding)
    """
    reg_write(REG_ALICE_PHASE_STAGED, alice_phase & 0x3)
    reg_write(REG_BOB_PHASE_STAGED, bob_phase & 0x3)
    reg_write(REG_PHASE_APPLY, 1)

def get_status():
    """Read and decode the STATUS register."""
    s = reg_read(REG_STATUS)
    return {
        'alice_running': bool(s & 0x1),
        'bob_running':   bool(s & 0x2),
        'sw_mode':       bool(s & 0x4),
    }

def get_active_phases():
    """Read back the currently active phase for each channel."""
    a = reg_read(REG_ALICE_PHASE_ACTIVE) & 0x3
    b = reg_read(REG_BOB_PHASE_ACTIVE) & 0x3
    return {'alice': PHASE_LABELS[a], 'bob': PHASE_LABELS[b]}

# Verify connectivity
version = reg_read(REG_VERSION)
print(f"VERSION: {version:#010x}")
assert version == 0x2026_0425, f"Unexpected version: {version:#010x}"

VERSION: 0x20260425


## 3. Configure RF Data Converter

DAC tile 229 (tile index 1), slices 0 and 2. NCO at 80 MHz, I/Q -> Real mixer mode.

In [12]:
dac_tileA = rf.dac_tiles[0]
dac_tileB = rf.dac_tiles[2]
print(f"DAC Tile 0: {dac_tileA}; PLL locked = {dac_tileA.PLLLockStatus}")
print(f"DAC Tile 2: {dac_tileB}; PLL locked = {dac_tileB.PLLLockStatus}")

blockA = dac_tileA.blocks[0]
print(f"\nAlice (slice 0):")
print(f"  Status: {blockA.BlockStatus}")
print(f"  Mixer:  {blockA.MixerSettings}")
blockA.MixerSettings['Freq'] = 80.0
blockA.UpdateEvent(xrfdc.EVENT_MIXER)
print(f"  Mixer (after update): {blockA.MixerSettings}")

blockB = dac_tileB.blocks[0]
print(f"\nBob (slice 0):")
print(f"  Status: {blockB.BlockStatus}")
print(f"  Mixer:  {blockB.MixerSettings}")
blockB.MixerSettings['Freq'] = 80.0
blockB.UpdateEvent(xrfdc.EVENT_MIXER)
print(f"  Mixer (after update): {blockB.MixerSettings}")

DAC Tile 0: <xrfdc.RFdcDacTile object at 0xffff431c5b70>; PLL locked = 2
DAC Tile 2: <xrfdc.RFdcDacTile object at 0xffff431e7a30>; PLL locked = 2

Alice (slice 0):
  Status: {'SamplingFreq': 6.4, 'AnalogDataPathStatus': 16, 'DigitalDataPathStatus': 8449, 'DataPathClocksStatus': 1, 'IsFIFOFlagsEnabled': 3, 'IsFIFOFlagsAsserted': 0}
  Mixer:  {'Freq': 79.99999999999545, 'PhaseOffset': 0.0, 'EventSource': 2, 'CoarseMixFreq': 16, 'MixerMode': 2, 'FineMixerScale': 0, 'MixerType': 2}
  Mixer (after update): {'Freq': 79.99999999999545, 'PhaseOffset': 0.0, 'EventSource': 2, 'CoarseMixFreq': 0, 'MixerMode': 2, 'FineMixerScale': 0, 'MixerType': 2}

Bob (slice 0):
  Status: {'SamplingFreq': 6.4, 'AnalogDataPathStatus': 16, 'DigitalDataPathStatus': 8449, 'DataPathClocksStatus': 1, 'IsFIFOFlagsEnabled': 3, 'IsFIFOFlagsAsserted': 0}
  Mixer:  {'Freq': 79.99999999999545, 'PhaseOffset': 0.0, 'EventSource': 2, 'CoarseMixFreq': 16, 'MixerMode': 2, 'FineMixerScale': 0, 'MixerType': 2}
  Mixer (after upda

## 4. Enable Outputs

Make sure SW3 is in register mode (SW3=1) on the board before running this cell.

In [11]:
# Enable global + Alice + Bob
reg_write(REG_CTRL, 0x07)

status = get_status()
print(f"Status: {status}")
assert status['alice_running'], "Alice not running — check CTRL and SW3"
assert status['bob_running'], "Bob not running — check CTRL and SW3"

Status: {'alice_running': True, 'bob_running': True, 'sw_mode': True}


## 5. Set Phases

Use `set_phases(alice, bob)` to atomically update both channels.

Phase encoding: `0` = 0, `1` = pi/2, `2` = pi, `3` = 3pi/2

In [44]:
# Example: Alice at pi/2, Bob at 0
set_phases(alice_phase=0, bob_phase=0)

phases = get_active_phases()
print(f"Active phases: {phases}")

Active phases: {'alice': '0', 'bob': '0'}


## 6. BB84 Key Exchange Demo

Cycle through all basis/bit combinations to verify the system.

In [ ]:
import time

# All 16 Alice x Bob phase combinations
for alice_phase in range(4):
    for bob_phase in range(4):
        set_phases(alice_phase, bob_phase)
        time.sleep(0.05)  # settle time
        phases = get_active_phases()
        status = get_status()
        print(f"Alice={phases['alice']:>5s}  Bob={phases['bob']:>5s}  "
              f"running=[A:{status['alice_running']} B:{status['bob_running']}]")

## 7. Disable Outputs

In [7]:
reg_write(REG_CTRL, 0x00)
print(f"Status: {get_status()}")

Status: {'alice_running': False, 'bob_running': False, 'sw_mode': False}
